# Assemble datasets from simulations
Combine data from simulations of different network architectures

In [1]:
import numpy as np
import pandas as pd
import os
import pickle
from tqdm import tqdm
from joblib import Parallel, delayed
import re

from stoch_sim_model import *

In [2]:
# Set parameters
sim_kind = 'agent'
reg_model = ''
runs = '-1-'
comment = "acute_all-sparse-reg"

d = '/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/'

sim_sum_list = []
parameters_nets = []
prim_diff_bias_list = []
#sec_diff_bias_list = []
cell_series_list = []
# lineage_diff_nets = []

In [3]:
# Figure out which jobs didn't run:
d_rerun = '/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/raw/'
run_list = [int(re.search('sim_batch_(.*?)\.', f).group(1)) for f in os.listdir(d_rerun) if 'sim_batch' in f and comment in f and runs in f]
out = [str(x) for x in [k for k in np.arange(0, 284)] if x not in run_list]
print(len(out))
print(' '.join((out)))

0



In [4]:
num_cpu = 150
file_list = [f for f in os.listdir(os.path.join(d, "raw")) if runs in f and comment in f and 'sim_batch' in f]
num_files = len(file_list)

def import_dict_func(f,d):
    
    file_path = os.path.join(os.path.join(d, "raw"), f)
    with open(file_path, 'rb') as filename:  
        import_dict = pickle.load(filename)

    parameters = np.array(import_dict["parameters"])
    sim_sum = np.array(import_dict["summary_stats"])

    out = np.hstack((parameters, sim_sum))

    return out

# create dataframe of infection response statistics
var_names = np.concatenate((param_names_for_df, stat_names_for_df))
# mean_df = pd.DataFrame(np.vstack(Parallel(n_jobs = num_cpu, batch_size = max(int(num_files/num_cpu),1))(delayed(import_dict_func)(f = file_name, d = d) 
#                                                                                                         for file_name in file_list)), 
#                        columns = [i for i in var_names]).groupby(Na_reg + NE_reg + EM_reg + EE_reg + vir_vars, as_index=False).mean()
full_df = pd.DataFrame(np.vstack(Parallel(n_jobs = num_cpu, batch_size = max(int(num_files/num_cpu),1))(delayed(import_dict_func)(f = file_name, d = d) 
                                                                                                        for file_name in file_list)), 
                       columns = [i for i in var_names])

# # Save datasets
full_df.to_pickle(os.path.join(d, "raw", "stacked_full_data"+runs+"runs"+'-'+comment)+'.pkl')

In [5]:
with pd.option_context('display.max_columns', None):
    display(full_df)

,S_0,I_0,b_I,d_S,d_I,d_IE,K_I,b_H,d_H,K_H,N_0,max_Na,b_myc,d_myc,myc_thresh,t_bind,t_unbind,t_Na_div,t_E_div,t_M_div,t_E_die,t_cycle,psi_myc_I,psi_myc_HI,psi_myc_HE,L0_Na,psi_NE_I,psi_NE_HI,psi_NE_HE,L0_NE,psi_EM_I,psi_EM_HI,psi_EM_HE,L0_EM,psi_Edie_I,psi_Edie_HI,psi_Edie_HE,L0_Edie,p_load,T_max_pI,T_min_pI,harm_pI,harm_pS,max_pE,T_pE_max,T_pE_start,max_eM,T_pEcyteM,T_pE_end,frac_cM,int_pHE,int_pHI,min_pS
0,10000000.0,1000.0,1.500000e-07,0.01,0.15,12.0,10000.0,1.0,2.0,100000.0,100.0,4.0,144.0,49.906597,1.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,0.0,0.0,0.0,4.0,-3.0,1.5,0.0,4.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,-4.0,6.817271e+06,8.72,0.0,1.170399e+07,687.438163,490.0,2.17,0.42,475.0,0.591053,0.00,0.509950,256.882251,500913.860030,1.504788e+05
1,10000000.0,1000.0,1.500000e-07,0.01,0.15,12.0,10000.0,1.0,2.0,100000.0,100.0,4.0,144.0,49.906597,1.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,0.0,0.0,0.0,4.0,-3.0,1.5,0.0,4.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,-2.0,6.817093e+06,8.72,0.0,1.170399e+07,743.565995,563.0,2.10,0.40,509.0,0.609941,0.00,0.490099,312.830835,500901.214627,1.504833e+05
2,10000000.0,1000.0,1.500000e-07,0.01,0.15,12.0,10000.0,1.0,2.0,100000.0,100.0,4.0,144.0,49.906597,1.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,0.0,0.0,0.0,4.0,-3.0,1.5,0.0,4.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,6.817944e+06,8.71,0.0,1.170512e+07,387.255517,372.0,1.83,0.25,266.0,0.320677,0.00,0.463840,241.809543,500962.442073,1.504624e+05
3,10000000.0,1000.0,1.500000e-07,0.01,0.15,12.0,10000.0,1.0,2.0,100000.0,100.0,4.0,144.0,49.906597,1.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,0.0,0.0,0.0,4.0,-3.0,1.5,0.0,4.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,2.0,6.818208e+06,8.70,0.0,1.170581e+07,249.459415,329.0,1.83,0.30,176.0,0.230284,0.00,0.486486,125.322751,500981.271704,1.504560e+05
4,10000000.0,1000.0,1.500000e-07,0.01,0.15,12.0,10000.0,1.0,2.0,100000.0,100.0,4.0,144.0,49.906597,1.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,0.0,0.0,0.0,4.0,-3.0,1.5,0.0,4.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,4.0,6.818157e+06,8.70,0.0,1.170560e+07,279.428684,330.0,1.69,0.31,185.0,0.219730,0.00,0.431421,124.946017,500977.406792,1.504571e+05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73124995,10000000.0,500000.0,1.000000e-01,0.01,0.01,12.0,100000000.0,1.0,2.0,100000.0,1000.0,4.0,144.0,49.906597,1.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,1.5,-1.5,-1.5,0.0,0.0,0.0,0.0,4.0,5.841762e+06,30.00,0.0,1.118375e+07,7315.502373,5435.0,1.98,0.20,1888.0,0.374094,0.00,0.159384,812.014933,151253.041339,5.191447e+06
73124996,10000000.0,500000.0,1.000000e-01,0.01,0.01,12.0,100000000.0,1.0,2.0,100000.0,1000.0,4.0,144.0,49.906597,1.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,1.5,-1.5,-1.5,2.0,0.0,0.0,0.0,-4.0,5.800224e+06,30.00,0.0,1.110300e+07,55497.428870,29753.0,3.20,0.20,30339.0,3.613907,5.16,0.178953,3437.486164,150102.034721,5.189793e+06
73124997,10000000.0,500000.0,1.000000e-01,0.01,0.01,12.0,100000000.0,1.0,2.0,100000.0,1000.0,4.0,144.0,49.906597,1.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,1.5,-1.5,-1.5,2.0,0.0,0.0,0.0,-2.0,5.826472e+06,30.00,0.0,1.115373e+07,23891.181854,13754.0,2.38,0.15,13245.0,1.194929,0.00,0.164183,2095.890438,150866.708435,5.192167e+06
73124998,10000000.0,500000.0,1.000000e-01,0.01,0.01,12.0,100000000.0,1.0,2.0,100000.0,1000.0,4.0,144.0,49.906597,1.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,1.5,-1.5,-1.5,2.0,0.0,0.0,0.0,0.0,5.841080e+06,30.00,0.0,1.118240e+07,8035.111063,5778.0,2.47,0.26,4372.0,0.427422,0.00,0.184478,936.005994,151236.442255,5.191501e+06


In [6]:
# Create additional variables
virs = np.unique(full_df[['I_0','d_I','K_I','b_I','K_H','N_0']].values, axis = 0)

full_df['antigenicity_over_harm'] = antigenicity_over_harm(full_df)
full_df['T_pE_clear'] = full_df['T_max_pI'] - full_df['T_pE_start']
full_df['max_eM_fold'] = np.log10(1 + full_df['max_eM']/full_df['N_0'])
full_df['stim_pI'] = np.log(1 + (full_df['p_load']/full_df['K_I']))
full_df['stim_pHI'] = np.log(1 + (full_df['int_pHI']/full_df['K_H']))
full_df['stim_pHE'] = np.log(1 + (full_df['int_pHE']/full_df['K_H']))
full_df['max_pE_fold'] = np.log10(1 + full_df['max_pE']/full_df['N_0'])
full_df['scaled_min_pS'] = full_df['min_pS']/full_df['S_0']
full_df['log_T_pEcyteM'] = np.log10(1 + full_df['T_pEcyteM']/sim_duration)

# identify Biologically evidenced networks
keep_vars = ['harm_pI', 'harm_pS', 'max_eM_fold', 'frac_cM', 'log_T_pEcyteM',
             'T_min_pI', 'T_max_pI', 'T_pE_start', 'T_pE_clear',
             'stim_pI', 'stim_pHI', 'stim_pHE',
             'scaled_min_pS', 'antigenicity_over_harm']

In [7]:
# save data sets
full_infection_scenarios = []
mean_of_infection_scenarios = []
std_of_infection_scenarios = []
no_eff_data = [[] for i in np.arange(len(virs))]
b_S = d_S*S_0

for l, (I_0, d_I, K_I, b_I, K_H, N_0) in enumerate(tqdm(virs)):
    data = full_df.loc[(full_df["d_I"] == d_I)*(full_df["K_I"] == K_I)*(full_df["b_I"] == b_I)*(full_df["K_H"] == K_H)*(full_df["N_0"] == N_0)*(full_df["I_0"] == I_0), 
    ['b_I','d_I', 'K_I', 'I_0','S_0', 'N_0', 'd_S', 'K_H'] + Na_reg + NE_reg + EM_reg + EE_reg + keep_vars]

    # compute infection harm without T cell response
    no_eff_data[l] = lin_stoch_sim(N_0 = 0, I_0 = I_0, K_I = K_I, d_I = d_I, b_I = b_I,
                                   infection_model = "cancer" if b_I >= b_C else "acute")
    no_eff_stats = no_eff_data[l]["summary_stats"]

    data.loc[:,"harm_pI_noprotection"] = no_eff_stats[3]/S_0
    data.loc[:,"peff_clearance"] = (no_eff_stats[3] - data['harm_pI'].to_numpy())/S_0
    data.loc[:,"peff_toxicity"] = data['harm_pS'].to_numpy()/S_0

    mean_of_infection_scenarios.append(data.groupby(Na_reg + NE_reg + EM_reg + EE_reg + vir_vars, as_index=False).mean())
    std_of_infection_scenarios.append(data.groupby(Na_reg + NE_reg + EM_reg + EE_reg + vir_vars, as_index=False).std())
    full_infection_scenarios.append(data)

# stack datasets
pd.concat(mean_of_infection_scenarios).to_pickle('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/mean/processed_data'+runs+'runs'+'-'+comment+'.pkl')
pd.concat(std_of_infection_scenarios).to_pickle('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/std/processed_data'+runs+'runs'+'-'+comment+'.pkl')
pd.concat(full_infection_scenarios).to_pickle('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/processed_full_data'+runs+'runs'+'-'+comment+'.pkl')

with open('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/mean/list_processed_data'+runs+'runs'+'-'+comment+'.pkl', 'wb') as f:
    pickle.dump(mean_of_infection_scenarios, f)

with open('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/std/list_processed_data'+runs+'runs'+'-'+comment+'.pkl', 'wb') as f:
    pickle.dump(std_of_infection_scenarios, f)

with open('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/list_processed_full_data'+runs+'runs'+'-'+comment+'.pkl', 'wb') as f:
    pickle.dump(full_infection_scenarios, f)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 234/234 [30:24<00:00,  7.80s/it]


In [8]:
# Clear memory
del full_df, mean_of_infection_scenarios, std_of_infection_scenarios, full_infection_scenarios